# 04 Scans, optimisation, matching

This notebook runs the five scripts of `04_scans_optimisation/` in order with `%run`, so it runs exactly the same code as the command line (`python 04_scans_optimisation/NN_name.py`). Each script's docstring says what it shows, what it needs and what it checks.

20 and 21 use a one-quad lattice built in Python (with `02_tracking/simple_lattice.py`) and check against matrix optics. 22–24 run JFEL from laura-lattices: the bunch compressor (22, 24) and the match into the undulator (23), checked against the JFEL README.

Everything here is tier A (pip only); 22 adds an elegant check when docker and the simcodes image are there. 22 takes about 80 s, the others under 20 s.

## 20 Measure the emittance with a quad scan

Vary one quad's `k1l` with `modifyElement`, track, and record $\sigma^2$ on a screen a distance $L$ downstream. With the quad as a thin lens of strength $k$, $\sigma^2 = A k^2 + B k + C$, and the parabola's coefficients give the beam's sigma matrix at the quad, so its emittance $\sqrt{\Sigma_{11}\Sigma_{22} - \Sigma_{12}^2}$. The exact fit uses the thick-quad matrices instead: $\sigma^2$ is linear in $\Sigma_{11}, \Sigma_{12}, \Sigma_{22}$.

The exact fit gives back the beam's emittance to $5\times10^{-6}$ in both planes. The thin-lens fit is 1% off in $x$, where the scan passes through the waist, but 43% off in $y$, where $\sigma_y$ only grows: there the fit is ill-conditioned and the small model error is amplified. A real scan should span the waist.

In [ ]:
%run 20_quad_scan_emittance.py

In [ ]:
from IPython.display import Image, display
display(Image(filename="../output/20_quad_scan_emittance/quad_scan.png"))

## 21 Optimise a setting with scipy

Wrap "set, track, read the screen" in a function and give it to `scipy.optimize.minimize_scalar`. It finds the quad strength for the smallest $\sigma_x$ on the screen in 10 tracks, the same as the thick-quad matrix model to $3\times10^{-6}$; the thin-lens answer $k = 1/L + \Sigma_{12}/\Sigma_{11}$ is 0.6% off. Every trial goes to a CSV, and `save_changes_file` keeps the best setting for `load_changes_file` later.

In [ ]:
%run 21_optimise_scipy.py

## 22 Optimise the JFEL compressor for 1000 A

`scipy.optimize.brentq` adjusts the L4H harmonic cavity's phase until the peak current after JFEL's chicane is 1000 A (Ocelot, structure wakes on): about $-180.84°$, in 7 tracks. The optimum sits on the edge of full compression, where $\pm 0.2°$ moves the current by 8%.

Then the JFEL README's warning, shown: the model must include what shapes the compression. Without the structure wakes, Ocelot and Cheetah agree with each other (2%), and both give half the current. An optimisation in a code without wakes would land somewhere else. elegant (if docker is there) shows the same wake effect.

In [ ]:
%run 22_optimise_compression.py

## 23 Match into the undulator with a `match:` block

JFEL's `jfel_match_undulator.def` has a `match:` block on its S13 line: four quads as variables, targets at the undulator entrance. Before tracking that line, SIMBA runs Ocelot's matcher and writes the quad strengths back. The target is the undulator's periodic Twiss, from a SIMBA line with `tracking: {periodic: True}` and `ring_parameters()`.

From half strength the matcher returns to the strengths stored in the lattice, and the beam reproduces the README's table. CSR is turned off, as in the README's match: on main the `.def` would include it (`ISSUES.md` #28).

In [ ]:
%run 23_match_with_def.py

## 24 A two-parameter scan in parallel

Chicane angle × L4H phase, 50 runs over a `ProcessPoolExecutor`. Each process builds one `Framework`, gives each run its own directory with `setSubDirectory`, and uses one thread (Ocelot's numpy threads would otherwise fight over the cores). The results go into a CSV. Straight-line fits to the two arms of each row of $\sigma_t$ find the phase of full compression: the README's "compression valley", rising steadily with the angle.

In [ ]:
%run 24_parameter_scan_parallel.py

In [ ]:
display(Image(filename="../output/24_parameter_scan_parallel/valley.png"))